In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import NuRadioReco.modules.io.eventReader
event_reader = NuRadioReco.modules.io.eventReader.eventReader()

def digitize(values, bits=14, v_min=-.5, v_max=.5):
    """Simulate digitization to N bits with given voltage range."""
    values = np.clip(values,v_min,v_max) # Clip
    # Quantize to integer levels
    levels = 2**bits - 1  # e.g., 16383 for 14-bit
    # Scale to [0, levels]
    scaled = (values - v_min) / (v_max - v_min) * levels
    # Round to nearest integer
    digitized = np.round(scaled).astype(int)
    # Convert back to voltage
    return digitized / levels * (v_max - v_min) + v_min

file = '/data/i3store/users/nmanic/classifier_consistency/ARZ2020/benchmark/nu/nur/nu_f000000_part02.nur'
event_reader.begin(file)
for iE, event in enumerate(event_reader.run()):

    primary = event.get_primary()

    for iStation, station in enumerate(event.get_stations()):

        # a fig and axes for our waveforms (4 rows, 2 columns)
        fig, axs = plt.subplots(1, 2, figsize=(12, 4))
        
        # this loops through "mock data" (with noise added, etc.) - channels 0-3
        ch = station.get_channel(0)
        #times=ch.get_times()-ch.get_times()[0]
        axs[0].plot(ch.get_times(), ch.get_trace())
        axs[0].set_title("Channel 0 (Noised)")
        axs[0].set_xlabel("Time (ns)")
        axs[0].set_ylabel("Voltage (V)")
        #axs[0].set_xlim(0,2400)

        ch = station.get_channel(40)
        times=ch.get_times()-ch.get_times()[0]
        axs[1].plot(times, ch.get_trace())
        axs[1].set_title("Channel 40 (Noiseless)")
        axs[1].set_xlabel("Time (ns)")
        axs[1].set_ylabel("Voltage (V)")
        axs[1].set_xlim(0,2400)

        fig.tight_layout()
        plt.show()

        print(len(ch.get_trace()))       # number of samples
        print(ch.get_sampling_rate())     # samples per ns (GHz-ish units)
        print(ch.get_times()[-1] - ch.get_times()[0])  # actual duration

        fig, axs = plt.subplots(4, 1, figsize=(8, 10))

        for i, ch_id in enumerate([0,1,2,3]):
            ch=station.get_channel(ch_id)
            times=ch.get_times()-ch.get_times()[0]
            axs[i].plot(times,ch.get_trace())
            axs[i].set_ylabel(f"Ch {ch_id} Voltage (V)")
            axs[i].set_xlabel("Time (ns)")
            axs[i].set_xlim(1590, 1660)
            fig.tight_layout()

        fig, axs=plt.subplots(1,2, figsize=(12,4), sharey=True)

        ch = station.get_channel(0)
        axs[0].plot(times,ch.get_trace())
        axs[1].plot(times,digitize(ch.get_trace()))
        axs[0].set_title("Undigitized Trace")
        axs[1].set_title("Digitized Trace")
        for ax in axs:
            ax.set_xlabel("Time (ns)")
            ax.set_ylabel("Voltage (V)")
            ax.set_xlim(1550,1660)

    if iE == 12:
        break
